# FSM Task Writing

**Purpose:** Validate FSM's core mechanism — that Claude Code reads tasks from `~/.claude/tasks/{session_id}/` and that writing JSON files there makes tasks appear in the agent's task list. This is the debug notebook: future FSM issues should start here.

Source: Self-driving — injects tasks via filesystem, verifies via `claude -p --session-id`

## Background

### Task File Location

Claude Code maintains a per-session task store at:
```
~/.claude/tasks/{session_id}/{task_id}.json
```

Each file is a single JSON object representing one task. Claude Code polls or reads this directory to populate the agent's task list.

### Session ID Source

The session ID is available in PostToolUse hook input as `hook_data['session_id']`. It is a UUID string identifying the active Claude Code session.

### Task File Schema

Required fields: `id`, `subject`, `description`, `status`, `blocks`, `blockedBy`  
Optional fields: `activeForm`, `owner`, `metadata`

```json
{
  "id": "42",
  "subject": "Imperative description of what to do",
  "description": "Detailed explanation with acceptance criteria",
  "status": "pending",
  "blocks": [],
  "blockedBy": [],
  "activeForm": "Present-continuous form shown while in_progress",
  "owner": "agent-name",
  "metadata": {}
}
```

Valid `status` values: `pending`, `in_progress`, `completed`, `deleted`

In [ ]:
# Task schema as a TypedDict — use this as the canonical reference when constructing task objects
from typing import TypedDict, Optional

class Task(TypedDict, total=False):
    # Required fields
    id: str
    subject: str
    description: str
    status: str        # 'pending' | 'in_progress' | 'completed' | 'deleted'
    blocks: list       # list of task IDs this task blocks
    blockedBy: list    # list of task IDs that block this task
    # Optional fields
    activeForm: str    # present-continuous label shown during in_progress
    owner: str         # agent name that claimed the task
    metadata: dict     # arbitrary key-value store

print("Task TypedDict defined.")
print("Required: id, subject, description, status, blocks, blockedBy")
print("Optional: activeForm, owner, metadata")

In [ ]:
# list_sessions() — lists session directories sorted by most recently modified
import os
from pathlib import Path
from datetime import datetime

TASKS_ROOT = Path.home() / '.claude' / 'tasks'

def list_sessions() -> list[Path]:
    """Return session directories sorted by mtime (most recent first)."""
    if not TASKS_ROOT.exists():
        return []
    dirs = [d for d in TASKS_ROOT.iterdir() if d.is_dir()]
    return sorted(dirs, key=lambda d: d.stat().st_mtime, reverse=True)

sessions = list_sessions()
if sessions:
    print(f"Found {len(sessions)} session(s):")
    for s in sessions[:5]:  # show up to 5
        mtime = datetime.fromtimestamp(s.stat().st_mtime).strftime('%Y-%m-%d %H:%M:%S')
        task_count = len(list(s.glob('*.json')))
        print(f"  {s.name}  ({task_count} task files, modified {mtime})")
else:
    print(f"No sessions found at {TASKS_ROOT}")

In [ ]:
# read_tasks() — reads all task JSON files for a given session
import json

def read_tasks(session_id: str) -> list[dict]:
    """Read all task files for a session. Returns list of task dicts sorted by id."""
    task_dir = TASKS_ROOT / session_id
    if not task_dir.exists():
        raise FileNotFoundError(f"Session directory not found: {task_dir}")
    tasks = []
    for f in task_dir.glob('*.json'):
        try:
            tasks.append(json.loads(f.read_text()))
        except json.JSONDecodeError as e:
            print(f"Warning: could not parse {f.name}: {e}")
    # Sort by numeric id if possible, else string sort
    def sort_key(t):
        try:
            return int(t.get('id', 0))
        except (ValueError, TypeError):
            return t.get('id', '')
    return sorted(tasks, key=sort_key)

# Demo: read tasks from the most recent session
if sessions:
    session_id = sessions[0].name
    tasks = read_tasks(session_id)
    print(f"Session: {session_id}")
    print(f"Tasks ({len(tasks)}):")
    for t in tasks:
        print(f"  [{t.get('status', '?')}] #{t.get('id')} — {t.get('subject', '(no subject)')}")
else:
    print("No sessions available to read tasks from.")

In [ ]:
# write_task() — writes a task file to a session directory (with basic validation)
import json
from pathlib import Path

REQUIRED_FIELDS = {'id', 'subject', 'description', 'status', 'blocks', 'blockedBy'}
VALID_STATUSES = {'pending', 'in_progress', 'completed', 'deleted'}

def write_task(session_id: str, task: dict) -> Path:
    """Write a task file. Returns the path written. Raises ValueError on invalid input."""
    missing = REQUIRED_FIELDS - set(task.keys())
    if missing:
        raise ValueError(f"Task missing required fields: {missing}")
    if task['status'] not in VALID_STATUSES:
        raise ValueError(f"Invalid status '{task['status']}'. Must be one of {VALID_STATUSES}")
    task_dir = TASKS_ROOT / session_id
    if not task_dir.exists():
        raise FileNotFoundError(f"Session directory not found: {task_dir}")
    task_file = task_dir / f"{task['id']}.json"
    task_file.write_text(json.dumps(task, indent=2))
    return task_file

print("write_task() defined. Required fields:", REQUIRED_FIELDS)

In [ ]:
# Demo: write task ID 9999, read it back, assert round-trip fidelity, then clean up
import json

if not sessions:
    print("Skipping demo — no sessions found.")
else:
    session_id = sessions[0].name
    TEST_TASK = {
        "id": "9999",
        "subject": "FSM round-trip test task",
        "description": "Written by fsm-task-writing.ipynb to validate the write/read cycle.",
        "status": "pending",
        "blocks": [],
        "blockedBy": []
    }

    # Write
    task_file = write_task(session_id, TEST_TASK)
    print(f"Written: {task_file}")

    # Read back
    read_back = json.loads(task_file.read_text())
    print(f"Read back: {read_back}")

    # Assert round-trip fidelity
    assert read_back == TEST_TASK, f"Round-trip mismatch!\nExpected: {TEST_TASK}\nGot: {read_back}"
    print("Round-trip assertion passed.")

    # Cleanup
    task_file.unlink()
    print(f"Cleaned up: {task_file.name} deleted.")

In [ ]:
# Agent visibility test — inject a task, then ask claude -p if it can see it
import json, uuid, shutil
from pathlib import Path
from lib import run_claude

TASKS_ROOT = Path.home() / '.claude' / 'tasks'

# Generate a session ID and pre-create the task directory
session_id = str(uuid.uuid4())
task_dir = TASKS_ROOT / session_id
task_dir.mkdir(parents=True, exist_ok=True)

# Write a test task
test_task = {
    "id": "1",
    "subject": "Notebook injection test",
    "description": "This task was injected by fsm-task-writing.ipynb to verify agent visibility.",
    "status": "pending",
    "blocks": [],
    "blockedBy": []
}
(task_dir / "1.json").write_text(json.dumps(test_task, indent=2))
print(f"Injected task into session: {session_id}")

# Run claude -p with the pre-created session
cr = run_claude("List your current tasks and describe each one.", session_id=session_id)

print("Agent response:")
print(cr.output.get("result", {}).get("content", [{}])[0].get("text", "(no text)"))

# Check if agent saw the injected task
saw_task = "injection test" in json.dumps(cr.output).lower()
print(f"\nAgent saw injected task: {saw_task}")

# Cleanup
shutil.rmtree(task_dir)
print(f"Cleaned up: {task_dir}")

## Sandbox Note

The macOS sandbox that Claude Code runs under blocks **hook scripts** from writing to `~/.claude/tasks/`. The `write-task.py` spike confirmed this — it caught a `PermissionError` when the hook tried to write to the task directory.

**What works:**
- Direct Python writes from notebooks (this notebook) ✅
- Pre-creating tasks + `claude -p --session-id` ✅ (NEW)
- Hook scripts reading from `~/.claude/tasks/` ✅

**What doesn't work:**
- Hook scripts writing to `~/.claude/tasks/` ❌ (sandbox blocks it)

**Implication for FSM:** Task injection from hooks remains blocked by the sandbox,
but external processes (notebooks, CI scripts) can inject tasks using `--session-id`.